In [ ]:
import pandas as pd
import numpy as np
import blinksear as ear
import blinksdistance as distance
import blinkscolors as colors
import json
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
def generate_webgazer_dataframe(df):
    df_webgazer = df[~df["webgazer_data"].isna()]
    dfs = []
    for row in range(len(df_webgazer)):
        json_data = df_webgazer["webgazer_data"].iloc[row].replace("'", "\"")
        data_list = json.loads(json_data)
        df_tmp = pd.DataFrame(data_list)
        df_tmp["trial"] = row + 1
        dfs.append(df_tmp)
    return pd.concat(dfs)

In [ ]:
def generate_chinrest_dataframe(df, i):
    df_chin = df[~df["view_dist_mm"].isna()][["view_dist_mm", "trial_index", "rt"]]
    df_chin["experiment"] = i
    return df_chin

In [ ]:
# This can load all data but is not storing df_webgazer in any place
files = Path("./data/all-exp-separados").glob("*.csv")
dfs = []
dfs_chin = []
i = 0
for file in files:
    if "blink" in str(file):
        print(file)
        df = pd.read_csv(file)
        df_tmp = generate_webgazer_dataframe(df)
        print(len(df_tmp))
        df_tmp["experiment"] = i
        df_tmp["file_name"] = str(file)
        dfs.append(df_tmp)
        df_virtual_chin_tmp = generate_chinrest_dataframe(df, i)
        dfs_chin.append(df_virtual_chin_tmp)
        #print(df_virtual_chin_tmp)
        i+=1

df_webgazer = pd.concat(dfs)
df_virtual_chin = pd.concat(dfs_chin)
print(len(df_webgazer))

In [ ]:
df_webgazer

In [ ]:
df_virtual_chin.head()

In [ ]:
df_virtual_chin.groupby("experiment").mean()

In [ ]:
# This can load all data but is not storing df_webgazer in any place
files = Path("./data/all-exp-separados").glob("*.csv")
dfs = []
dfs_chin = []
i = 0
for file in files:
    if "blink" in str(file):
        print(file)
        df = pd.read_csv(file)
        df_tmp = generate_webgazer_dataframe(df)
        print(len(df_tmp))
        df_tmp["experiment"] = i
        df_tmp["file_name"] = str(file)
        dfs.append(df_tmp)
        df_virtual_chin_tmp = generate_chinrest_dataframe(df, i)
        dfs_chin.append(df_virtual_chin_tmp)
        #print(df_virtual_chin_tmp)
        i+=1

df_webgazer = pd.concat(dfs)
df_virtual_chin = pd.concat(dfs_chin)
print(len(df_webgazer))

In [ ]:
files = Path("./data/all-exp-separados").glob("*.csv")

for file in files:
    if "blink" in str(file):
        df = pd.read_csv(file)
        df_webgazer = generate_webgazer_dataframe(df)

        df_virtual_chin = df[~df["view_dist_mm"].isna()]
        df_virtual_chin.loc[:, ["view_dist_centimeters"]] = (
            df[~df["view_dist_mm"].isna()]["view_dist_mm"] / 10
        )
        (
            df_virtual_chin["rt"].reset_index(drop=True),
            df_webgazer.groupby("trial")["t"].max(),
        )
df_virtual_chin["rt"].head()



In [ ]:
files = Path("./data/all-exp-separados").glob("*.csv")

for file in files:
    if "blink" in str(file):
        df = pd.read_csv(file)
        df_webgazer = generate_webgazer_dataframe(df)

        df_virtual_chin = df[~df["view_dist_mm"].isna()].copy()
        df_virtual_chin.loc[:, ["view_dist_centimeters"]] = (
            df[~df["view_dist_mm"].isna()]["view_dist_mm"] / 10
        )
        (
            df_virtual_chin["rt"].reset_index(drop=True),
            df_webgazer.groupby("trial")["t"].max(),
        )
df_webgazer

In [ ]:
files = Path("./data/all-exp-separados").glob("*.csv")

for file in files:
    if "blink" in str(file):
        df = pd.read_csv(file)
        df_webgazer = generate_webgazer_dataframe(df)

        df_virtual_chin = df[~df["view_dist_mm"].isna()].copy()
        df_virtual_chin.loc[:, ["view_dist_centimeters"]] = (
            df[~df["view_dist_mm"].isna()]["view_dist_mm"] / 10
        )
        (
            df_virtual_chin["rt"].reset_index(drop=True),
            df_webgazer.groupby("trial")["t"].max(),
        )

        df_virtual_chin['t'] = (
            df_virtual_chin['rt'] + 
            df_virtual_chin['rt'].cumsum().shift(1).fillna(0)
        )
        max_por_trial = df_webgazer.groupby('trial')['t'].max()
        max_acumulado_anterior = max_por_trial.cumsum().shift(1)
        df_webgazer['t'] = df_webgazer['t'] + df_webgazer['trial'].map(max_acumulado_anterior).fillna(0)
        df_webgazer = df_webgazer.astype({"dz": float, "t": float})
        df_webgazer.plot(x="t", y="dz", alpha=0.75, fontsize=12)
        plt.xlabel("tiempo (ms)")
        plt.ylabel("dz (cms)")
        plt.xlim(0, df_webgazer["t"].max() + 1500)
        plt.hlines(
            df_webgazer["dz"].mean(),
            df_webgazer["t"].min(),
            df_webgazer["t"].max(),
            color="k",
            linestyle="--",
            lw=2,
            label="dz promedio",
        )
        colors = ["r", "k", "g"]
        for i, (virtual_chin_time, virtual_chin_distance) in enumerate(
            zip(df_virtual_chin["t"], df_virtual_chin["view_dist_centimeters"])
        ):
            plt.axvline(virtual_chin_time, color="red", linestyle="--", lw=1)
            plt.scatter(
                virtual_chin_time,
                virtual_chin_distance,
                s=50,
                color=colors[i],
                label=f"virtual chin rest trial {i + 1}",
            )
            #plt.ylim(
              #  min(df_webgazer["dz"].min(), virtual_chin_distance) - 10,
             #   max(virtual_chin_distance, df_webgazer["dz"].max()) + 10,
            #)
            plt.ylim(
                20,
                140,
            )

        #plt.title(file)
        plt.title(file)
        plt.rcParams['font.size'] = 14
        plt.legend(loc="upper left", fontsize=10)
        plt.show()

        plt.figure(figsize=(8, 6))
        box_plot = plt.boxplot(df_webgazer["dz"].dropna(), patch_artist=True)

      # Personalizar colores
        box_plot['boxes'][0].set_facecolor('lightblue')
        box_plot['boxes'][0].set_alpha(1)
        box_plot['boxes'][0]

        plt.ylim(
            20,
            140,
        )
        
        plt.title(file, fontsize=14)
        plt.xlabel("", fontsize=12)
        plt.ylabel("dz(cm)", fontsize=12)
        plt.show()


In [ ]:
df_virtual_chin.head()

In [ ]:
files = Path("./data/all-exp-separados").glob("*.csv")

for file in files:
    if ("blink" in str(file)) & (("18" in str(file)) | ("10" in str(file)) | ("_9" in str(file))):
        df = pd.read_csv(file)
        df_webgazer = generate_webgazer_dataframe(df)

        df_virtual_chin = df[~df["view_dist_mm"].isna()].copy()
        df_virtual_chin.loc[:, ["view_dist_centimeters"]] = (
            df[~df["view_dist_mm"].isna()]["view_dist_mm"] / 10
        )
        (
            df_virtual_chin["rt"].reset_index(drop=True),
            df_webgazer.groupby("trial")["t"].max(),
        )

        df_virtual_chin['t'] = (
            df_virtual_chin['rt'] + 
            df_virtual_chin['rt'].cumsum().shift(1).fillna(0)
        )
        max_por_trial = df_webgazer.groupby('trial')['t'].max()
        max_acumulado_anterior = max_por_trial.cumsum().shift(1)
        df_webgazer['t'] = df_webgazer['t'] + df_webgazer['trial'].map(max_acumulado_anterior).fillna(0)
        df_webgazer = df_webgazer.astype({"dz": float, "t": float})
        df_webgazer.plot(x="t", y="dz", alpha=0.75, fontsize=12)
        plt.xlabel("tiempo (ms)")
        plt.ylabel("dz (cms)")
        plt.xlim(0, df_webgazer["t"].max() + 1500)
        plt.hlines(
            df_webgazer["dz"].mean(),
            df_webgazer["t"].min(),
            df_webgazer["t"].max(),
            color="k",
            linestyle="--",
            lw=2,
            label="dz promedio",
        )
        colors = ["r", "k", "g"]
        for i, (virtual_chin_time, virtual_chin_distance) in enumerate(
            zip(df_virtual_chin["t"], df_virtual_chin["view_dist_centimeters"])
        ):
            plt.axvline(virtual_chin_time, color="red", linestyle="--", lw=1)
            plt.scatter(
                virtual_chin_time,
                virtual_chin_distance,
                s=50,
                color=colors[i],
                label=f"virtual chin rest trial {i + 1}",
            )
            #plt.ylim(
              #  min(df_webgazer["dz"].min(), virtual_chin_distance) - 10,
             #   max(virtual_chin_distance, df_webgazer["dz"].max()) + 10,
            #)
            plt.ylim(
                20,
                140,
            )

        #plt.title(file)
        plt.title(file)
        plt.rcParams['font.size'] = 14
        plt.legend(loc="upper left", fontsize=10)
        plt.show()

        plt.figure(figsize=(8, 6))
        box_plot = plt.boxplot(df_webgazer["dz"].dropna(), patch_artist=True)

      # Personalizar colores
        box_plot['boxes'][0].set_facecolor('lightblue')
        box_plot['boxes'][0].set_alpha(1)
        box_plot['boxes'][0]

        plt.ylim(
            20,
            140,
        )
        
        plt.title(file, fontsize=14)
        plt.xlabel("", fontsize=12)
        plt.ylabel("dz(cm)", fontsize=12)
        plt.show()
